In [62]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import swish
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math

In [63]:
df = pd.read_csv("/content/Housing.csv")
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
category_mappings = {}
def convert_to_int(column):
    if column.dtype == 'object':
        unique_vals = column.unique()
        val_map = {val: idx for idx, val in enumerate(unique_vals)}
        category_mappings[column.name] = val_map
        return column.map(val_map)
    return column

df = df.apply(convert_to_int)
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,0,0,0,0,0,2,0,0
1,12250000,8960,4,4,4,0,0,0,0,0,3,1,0
2,12250000,9960,3,2,2,0,0,1,0,1,2,0,1
3,12215000,7500,4,2,2,0,0,1,0,0,3,0,0
4,11410000,7420,4,1,2,0,1,1,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,0,0,1,0,1,2,1,2
541,1767150,2400,3,1,1,1,0,0,0,1,0,1,1
542,1750000,3620,2,1,1,0,0,0,0,1,0,1,2
543,1750000,2910,3,1,1,1,0,0,0,1,0,1,0


In [66]:
X = df.drop(columns=['price'])
y = df['price']

In [67]:
scaler = StandardScaler()
numerical_features = X_train.select_dtypes(include=np.number).columns
X_train_scaled = scaler.fit_transform(X_train[numerical_features])
X_test_scaled = scaler.transform(X_test[numerical_features])

In [68]:
X_train = pd.DataFrame(X_train_scaled, index=X_train.index, columns=numerical_features)
X_test = pd.DataFrame(X_test_scaled, index=X_test.index, columns=numerical_features)

X_train = pd.concat([X_train, X_train.select_dtypes(exclude=np.number)], axis=1)
X_test = pd.concat([X_test, X_test.select_dtypes(exclude=np.number)], axis=1)


In [69]:
# Build ANN model
model = Sequential([
    Dense(18, activation=swish, input_shape=(X_train.shape[1],)),
    Dense(26, activation=swish),
    Dense(20, activation=swish),
    Dense(15, activation=swish),
    Dense(1, activation='linear')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [70]:
model.compile(optimizer='rmsprop', loss='mse')

In [71]:
history = model.fit(X_train, y_train, epochs=200, batch_size=64, validation_data=(X_test, y_test))
model.save("housing_price_model.h5")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 24417593720832.0000 - val_loss: 30129992499200.0000
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 26530474360832.0000 - val_loss: 30129988304896.0000
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 26565744263168.0000 - val_loss: 30129988304896.0000
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 24577996488704.0000 - val_loss: 30129982013440.0000
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 24680064876544.0000 - val_loss: 30129977819136.0000
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 25306396098560.0000 - val_loss: 30129963139072.0000
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 26350790377472.0000 - val_loss: 30129954750464.0000
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 25636718510080.0000 - val_loss: 30129940070400.0000
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 24899796074496.0000 - val_loss: 30129917001728.0000
Epoch 10/20

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [79]:
train_rmse = math.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = math.sqrt(mean_squared_error(y_test, y_test_pred))
train_mse=mean_squared_error(y_train, y_train_pred)
test_mse=mean_squared_error(y_test, y_test_pred)
print(f"Training RMSE: {train_rmse}")
print(f"Testing RMSE: {test_rmse}")
print(f"Training MSE: {train_mse}")
print(f"Testing MSE: {test_mse}")

Training RMSE: 4019740.6617243355
Testing RMSE: 4445677.809014954
Training MSE: 16158314987520.0
Testing MSE: 19764051181568.0


In [73]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import mse
loaded_model=load_model('/content/housing_price_model.h5', custom_objects={'mse': mse})
new_data=pd.DataFrame([{
    'area':1500,
    'bedrooms':2,
    'bathrooms':2,
    'stories':2,
    'mainroad':1,
    'guestroom':1,
    'basement':1,
    'hotwaterheating':1,
    'airconditioning':1,
    'parking':2,
    'prefarea':1,
    'furnishingstatus':1
}])

In [74]:
new_data_scaled=scaler.transform(new_data[numerical_features])
prediction = loaded_model.predict(new_data_scaled)
print("Predicted price:", prediction[0, 0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Predicted price: 630572.6
